In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.3 Quantization

> 🎯 **Goal:** Shrink the model by storing weights in fewer bits, and measure the accuracy cost rather than assuming it.

**Why this matters.** Memory accounting (last lesson) showed weights are a fixed but large cost. Quantization is the direct lever on that cost: store each weight in 8 or 4 bits instead of 32 and you cut weight memory by 4x to 8x. That can be the difference between a model fitting on a cheaper GPU or not. But cheaper precision can cost quality, so the discipline is to measure the trade, not hope for it.

**The intuition.** A weight is just a number. Storing it in 32 bits is like writing a price with many decimal places when the nearest cent is fine. Quantization rounds each number to a coarser grid so it fits in fewer bits. Most of the time the model barely notices, because neural nets are robust to small perturbations. But round too coarsely and the errors accumulate, so you check.

**The mechanics.** `fake_quantize` simulates low-bit storage: it rounds each weight to the nearest value representable in `bits` bits (within the tensor's range) and keeps it as a float, so we can run the model normally and read off the quality. The experiment is a clean before/after: measure validation loss on the fp32 model, deep-copy it, quantize every parameter to 4 bits, and measure loss again with the *same* random seed so the only thing that changed is precision. The printed delta is the real, measured cost of going to 4-bit on this model. **Lower loss is better**, so a small positive delta means a small quality loss.

In [ ]:
import copy
from pocketlm import train_tiny, pick_config, fake_quantize, estimate_loss

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
g = torch.Generator().manual_seed(0)
before = estimate_loss(model, data, model.cfg.block_size, 16, iters=10, generator=g)
qmodel = copy.deepcopy(model)
with torch.no_grad():
    for p in qmodel.parameters():
        p.copy_(fake_quantize(p, bits=4))
g2 = torch.Generator().manual_seed(0)
after = estimate_loss(qmodel, data, model.cfg.block_size, 16, iters=10, generator=g2)
print(f"loss fp32: {before:.3f}  ->  4-bit: {after:.3f}  (delta {after - before:+.3f})")

**What just happened.** You have a concrete number: fp32 loss, 4-bit loss, and the delta between them. Four-bit weights take a quarter of the fp32 memory, and the printed delta is the price you paid in quality on this tiny model. On real large models the pattern holds: 8-bit is often nearly free, and 4-bit is a deliberate trade you make with eyes open, always backed by a measured number exactly like this one.

⚠️ **Common pitfalls**
- Reporting the memory saving but never the accuracy cost. The saving is the easy half, the trade is the point.
- Using different seeds for the before and after measurement, so you cannot tell quantization noise from sampling noise.
- Quantizing tiny or unusually sensitive tensors (like norms or embeddings) the same way as big linear layers. Real toolchains often keep some tensors in higher precision.

🏋️ **Try it yourself.** Change `bits=4` to `bits=8` and rerun. The delta should shrink, often close to zero, showing 8-bit is the safer default. Then try `bits=2` to see the quality fall off a cliff and understand why nobody ships 2-bit naively.

In [ ]:
import math
assert math.isfinite(after)
assert after > 0